In [1]:
import os

if not os.path.exists("/content/dl-soundstream-codec"):
    !git clone https://github.com/MathUmkaaa/dl-soundstream-codec.git

%cd /content/dl-soundstream-codec

!pip install -q gdown soundfile pystoi torchmetrics
!python scripts/download_checkpoint.py



Cloning into 'dl-soundstream-codec'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 135 (delta 33), reused 110 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 431.81 KiB | 6.35 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/dl-soundstream-codec
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.3 MB/s eta 0:00:00
Downloading...
From (original): https://drive.google.com/uc?id=1sTm7sgaQnJE0vV7g5NOiKzUF5WGa7nzg
From (redirected): https://drive.google.com/uc?id=1sTm7sgaQnJE0vV7g5NOiKzUF5WGa7nzg&confirm=t&uuid=b926993d-dc69-4854-a563-fef7cf00eb3d
To: /content/dl-soundstream-codec/checkpoints/model_best.pth
100% 124M/124M [00:01<00:00, 76.3MB/s]


In [2]:
import torch
import torchaudio

print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("cuda:", torch.cuda.is_available())

torch: 2.10.0+cu128
torchaudio: 2.10.0+cu128
cuda: True


In [4]:
import urllib.request

AUDIO_TEST = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

audio_path = "/tmp/input.wav"
urllib.request.urlretrieve(AUDIO_TEST, audio_path)


('/tmp/input.wav', <http.client.HTTPMessage at 0x7a018c3ae3f0>)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cuda


In [7]:
from src.model import SoundStream

model = SoundStream()
model = model.to(device)

checkpoint = torch.load("checkpoints/model_best.pth", map_location=device, weights_only=False)
checkpoint = checkpoint["state_dict"]

model.load_state_dict(checkpoint)
model.eval()


SoundStream(
  (encoder): Encoder(
    (layers): Sequential(
      (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,))
      (1): EncoderBlock(
        (layers): Sequential(
          (0): ResidualUnit(
            (layers): Sequential(
              (0): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(3,))
              (1): ELU(alpha=1.0)
              (2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
              (3): ELU(alpha=1.0)
            )
          )
          (1): ResidualUnit(
            (layers): Sequential(
              (0): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
              (1): ELU(alpha=1.0)
              (2): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
              (3): ELU(alpha=1.0)
            )
          )
          (2): ResidualUnit(
            (layers): Sequential(
              (0): Conv1d(32, 32, kernel_size=(7,), stride=(1,), padding=(27,), dilation=(9,))
              (1): ELU(alpha=1.0)
     

In [8]:
import torch.nn.functional as F

wav, sr = torchaudio.load(audio_path)
print(wav.shape, sr)

if wav.shape[0] > 1:
    wav = wav.mean(dim=0, keepdim=True)

if sr != 16000:
    wav = torchaudio.functional.resample(wav, sr, 16000)
    sr = 16000

pad = (-wav.shape[-1]) % 200
if pad > 0:
    wav = F.pad(wav, (0, pad), mode="replicate")


torch.Size([1, 185146]) 22050


In [9]:
x = wav.unsqueeze(0).to(device)

with torch.no_grad():
    recon1, recon2,recon3 = model(x)

orig = wav.squeeze(0).cpu()
recon1 = recon1.squeeze(0).squeeze(0).cpu()


In [12]:
from IPython.display import Audio

print("Original")
Audio(orig.numpy(), rate=16000)


Original


In [13]:
print("Reconstructed")
Audio(recon1.numpy(), rate=16000)

Reconstructed
